# Stage 1A T4/FP16 hardware-adapted reproduction

This unexecuted notebook is a thin Colab orchestrator for the tracked T4/FP16 path. It does not alter or replace the official BF16 notebook. A passing run establishes pinned API/runtime functionality only; native-BF16 reference reproduction remains pending.

Select the Colab `2025.07` Python 3.11 GPU runtime, add the Colab secret `HF_TOKEN`, and paste the final 40-character `stage-1a-t4-fp16` commit in the parameter cell. The token is passed only in the runner child environment and is never printed, serialized, inspected, or included in the ZIP.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shlex
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/eokahya/counterfactual-susceptibility.git"
PROJECT_REF = "stage-1a-t4-fp16"
EXPECTED_PROJECT_COMMIT = ""  # @param {type:"string"}
UPSTREAM_COMMIT = "8f1e2438df612464e229e44c4a00ff637bf9379b"
MODEL_REVISION = "c5ebcd40d208330abc697524c919956e692655cf"
TRANSCODER_REVISION = "bd5773156dea09893636c801df1237d0410307d2"
REPOSITORY_DIR = Path("/content/counterfactual-susceptibility")
VENV_DIR = Path("/content/cfsus-stage1a-t4-venv")
BUNDLE = Path("/content/stage1a-t4-fp16-small-artifacts.zip")

for revision in (
    EXPECTED_PROJECT_COMMIT,
    UPSTREAM_COMMIT,
    MODEL_REVISION,
    TRANSCODER_REVISION,
):
    if re.fullmatch(r"[0-9a-f]{40}", revision) is None:
        raise ValueError("Every expected revision must be a 40-character SHA")


def run(
    argv: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    print("+", shlex.join(argv))
    return subprocess.run(argv, cwd=cwd, env=env, check=check, text=True)


def capture(argv: list[str], *, cwd: Path | None = None) -> str:
    return subprocess.run(
        argv, cwd=cwd, check=True, text=True, capture_output=True
    ).stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Stage 1A requires Python 3.11; select a compatible Colab runtime"
    )
smi = capture(["nvidia-smi"])
cuda_match = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi)
if cuda_match is None:
    raise RuntimeError("nvidia-smi omitted CUDA driver compatibility")
driver_cuda = tuple(int(part) for part in cuda_match.groups())
if driver_cuda < (12, 4):
    raise RuntimeError("The pinned cu124 wheel requires driver CUDA >=12.4")
driver_line = capture(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,memory.free",
        "--format=csv,noheader,nounits",
    ]
)
driver_fields = [field.strip() for field in driver_line.split(",")]
if len(driver_fields) != 4:
    raise RuntimeError("nvidia-smi returned unexpected GPU metadata")
print(
    json.dumps(
        {
            "python": sys.version.split()[0],
            "gpu_name": driver_fields[0],
            "driver_version": driver_fields[1],
            "driver_cuda_compatibility": list(driver_cuda),
            "reported_total_mib": int(driver_fields[2]),
            "reported_free_mib": int(driver_fields[3]),
            "gpu_uuid_recorded": False,
        },
        sort_keys=True,
    )
)

In [ ]:
if REPOSITORY_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite checkout: {REPOSITORY_DIR}")
run(
    [
        "git",
        "clone",
        "--branch",
        PROJECT_REF,
        "--single-branch",
        REPOSITORY,
        str(REPOSITORY_DIR),
    ]
)
PROJECT_COMMIT = capture(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR)
if PROJECT_COMMIT != EXPECTED_PROJECT_COMMIT:
    raise RuntimeError(
        "Branch HEAD differs from EXPECTED_PROJECT_COMMIT; refusing mutable code"
    )
if capture(["git", "status", "--porcelain"], cwd=REPOSITORY_DIR):
    raise RuntimeError("Fresh project checkout is unexpectedly dirty")
print(f"Resolved {PROJECT_REF} to {PROJECT_COMMIT}")

In [ ]:
PLANNED_REQUIREMENTS = (
    REPOSITORY_DIR / "environments/stage1a/requirements-colab-py311-cu124-planned.txt"
)
requirements_text = PLANNED_REQUIREMENTS.read_text(encoding="utf-8")
required_pins = (
    "torch==2.6.0",
    "transformer-lens==3.2.1",
    "transformers==4.57.3",
    "nnsight==0.6.1",
    "huggingface-hub==0.36.2",
    f"circuit-tracer.git@{UPSTREAM_COMMIT}",
)
missing_pins = [pin for pin in required_pins if pin not in requirements_text]
if missing_pins:
    raise RuntimeError(f"Tracked dependency plan lacks pins: {missing_pins}")
system_package_pins = {
    "python3.11-venv": "3.11.15-1+jammy1",
    "python3-pip-whl": "22.0.2+dfsg-1ubuntu0.7",
    "python3-setuptools-whl": "68.1.2-2~jammy3",
}
run(["apt-get", "update"])
run(
    [
        "apt-get",
        "install",
        "--yes",
        "--no-install-recommends",
        *(f"{package}={version}" for package, version in system_package_pins.items()),
    ]
)
installed_system_packages = {
    package: capture(["dpkg-query", "-W", "-f=${Version}", package])
    for package in system_package_pins
}
if installed_system_packages != system_package_pins:
    raise RuntimeError(f"Pinned system package mismatch: {installed_system_packages}")
if VENV_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite environment: {VENV_DIR}")
run([sys.executable, "-m", "venv", str(VENV_DIR)])
PYTHON = str(VENV_DIR / "bin/python")
run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
        "torch==2.6.0",
    ]
)
run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://pypi.org/simple",
        "--requirement",
        str(PLANNED_REQUIREMENTS),
    ]
)
run(
    [PYTHON, "-m", "pip", "install", "--no-deps", "--editable", "."], cwd=REPOSITORY_DIR
)
run([PYTHON, "-m", "pip", "check"])

In [ ]:
verification_code = f'''
import importlib.metadata as metadata
import json
import torch

expected = {{
    "transformer-lens": "3.2.1",
    "transformers": "4.57.3",
    "nnsight": "0.6.1",
    "huggingface-hub": "0.36.2",
}}
for package, version in expected.items():
    if metadata.version(package) != version:
        raise RuntimeError(f"{{package}} version mismatch")
if metadata.version("torch").split("+", 1)[0] != "2.6.0":
    raise RuntimeError("PyTorch version mismatch")
if torch.version.cuda != "12.4" or not torch.cuda.is_available():
    raise RuntimeError("Expected available PyTorch CUDA 12.4 runtime")
properties = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
record = {{
    "gpu_name": properties.name,
    "compute_capability": [properties.major, properties.minor],
    "total_vram_bytes": total_bytes,
    "free_vram_bytes_before_load": free_bytes,
    "minimum_total_vram_bytes": 14 * 1024**3,
    "minimum_free_vram_bytes": 12 * 1024**3,
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
    "gpu_uuid_recorded": False,
}}
print(json.dumps({{"t4_fp16_preflight": record}}, sort_keys=True))
if "T4" not in properties.name or (properties.major, properties.minor) != (7, 5):
    raise RuntimeError("This hardware-adapted path requires an NVIDIA T4")
if record["bf16_supported"]:
    raise RuntimeError("T4 provenance unexpectedly reports native BF16 support")
if (
    total_bytes < record["minimum_total_vram_bytes"]
    or free_bytes < record["minimum_free_vram_bytes"]
):
    raise RuntimeError("Preflight requires >=14 GiB total and >=12 GiB free VRAM")
x = torch.ones((2, 2), device="cuda", dtype=torch.float16)
y = x @ x
if not torch.isfinite(y).all().item():
    raise RuntimeError("CUDA float16 execution probe was non-finite")
torch.cuda.synchronize()
distribution = metadata.distribution("circuit-tracer")
direct_url = json.loads(distribution.read_text("direct_url.json"))
vcs = direct_url.get("vcs_info", {{}})
if direct_url.get("url") != "https://github.com/decoderesearch/circuit-tracer.git":
    raise RuntimeError("Unexpected circuit-tracer source")
if vcs.get("commit_id") != "{UPSTREAM_COMMIT}" or vcs.get("vcs") != "git":
    raise RuntimeError("Unexpected circuit-tracer commit provenance")
'''
run([PYTHON, "-c", verification_code])

In [ ]:
try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("This cell must run in Google Colab") from exc

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add an accessible Colab secret named HF_TOKEN")
run_env = os.environ.copy()
run_env.update(
    {
        "HF_TOKEN": hf_token,
        "HUGGING_FACE_HUB_TOKEN": hf_token,
        "HF_HOME": "/content/cfsus-hf-cache",
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "TOKENIZERS_PARALLELISM": "false",
    }
)
runner_command = [
    PYTHON,
    "scripts/stage1a/run_stage1a_t4_fp16.py",
    "--config",
    "configs/stage1a_gemma2_2b_t4_fp16_reproduction.yaml",
    "--allow-download",
]
try:
    runner = run(runner_command, cwd=REPOSITORY_DIR, env=run_env, check=False)
finally:
    run_env.pop("HF_TOKEN", None)
    run_env.pop("HUGGING_FACE_HUB_TOKEN", None)
    hf_token = None

manifest_path = (
    REPOSITORY_DIR / "results/stage1a_t4_fp16/stage1a_t4_fp16_run_manifest.json"
)
if manifest_path.is_file():
    final_status = json.loads(manifest_path.read_text(encoding="utf-8"))["status"]
    print(f"Final T4/FP16 status: {final_status}")
else:
    final_status = "failed_before_manifest"
    print(f"Final T4/FP16 status: {final_status}")
if runner.returncode != 0:
    raise RuntimeError("Tracked T4 runner did not complete; inspect sanitized status")

In [ ]:
run(
    [
        PYTHON,
        "scripts/stage1a/validate_t4_fp16_artifacts.py",
        "--artifact-dir",
        "results/stage1a_t4_fp16",
        "--bundle",
        str(BUNDLE),
    ],
    cwd=REPOSITORY_DIR,
)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(f"Small artifact bundle: {BUNDLE}")
print(f"Bundle SHA-256: {sha256_file(BUNDLE)}")
print(f"Final status: {manifest['status']}")
print("Native-BF16 reference reproduction remains pending.")